# Sakura-Galtransl-14B-v3.8-Q5_K_S Local Colab Batch Translation
Run this notebook in Google Colab to deploy the Sakura translation model and run the EPUB translator directly inside Colab. No internal tunneling is needed.

## 1. Setup Environment (llama.cpp & EPUB-Translator)

In [ ]:
!echo "Downloading pre-compiled llama-server..."
import urllib.request, json, os, zipfile, tarfile
# 抓取官方最新 Releases
req = urllib.request.Request('https://api.github.com/repos/ggerganov/llama.cpp/releases', headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as r:
    releases = json.loads(r.read())

# 寻找适用于 Colab (Ubuntu + CUDA 12) 的现成二进制文件（支持 cu12, cuda12, cuda-12 等命名的 tar.gz 或 zip 压缩包）
asset_url = None
asset_name = ""
for release in releases:
    assets = release.get('assets', [])
    for asset in assets:
        name = asset['name'].lower()
        if 'ubuntu' in name and ('cu12' in name or 'cuda12' in name or 'cuda-12' in name) and not name.startswith('cudart'):
            if name.endswith('.zip') or name.endswith('.tar.gz'):
                # 优先选用包含 bin 和 cuda-12 的核心可执行打包包
                asset_url = asset['browser_download_url']
                asset_name = asset['name']
                break
    if asset_url:
        break

if asset_url:
    print(f"Found pre-compiled binary: {asset_url}\nDownloading {asset_name}...")
    local_file = asset_name
    urllib.request.urlretrieve(asset_url, local_file)
    
    os.makedirs("llama_prebuilt", exist_ok=True)
    if local_file.endswith('.zip'):
        with zipfile.ZipFile(local_file, "r") as z:
            z.extractall("llama_prebuilt")
    elif local_file.endswith('.tar.gz'):
        with tarfile.open(local_file, "r:gz") as t:
            t.extractall("llama_prebuilt")
            
    os.system("chmod -R +x llama_prebuilt")
    print("Extraction complete!")
else:
    raise FileNotFoundError("Error: No compatible pre-compiled llama-server binary found for Colab (Ubuntu + CUDA 12) in GitHub releases. Please check the GitHub releases page for available assets.")

!echo "Downloading Sakura GGUF model..."
import huggingface_hub
huggingface_hub.hf_hub_download(repo_id="SakuraLLM/Sakura-GalTransl-14B-v3.8", filename="Sakura-Galtransl-14B-v3-Q5_K_S.gguf", local_dir=".", local_dir_use_symlinks=False)

!echo "Cloning and installing EPUB-Bilingual-Translator..."
!git clone https://github.com/yukari502/EPUB-Bilingual-Translator.git
!cd EPUB-Bilingual-Translator && pip install -e .

## 2. Start Model Server (Background)

In [ ]:
import subprocess
import time
import os
import glob

# 在解压的文件夹中寻找 llama-server 执行文件
server_bin = None
for path in glob.glob("./llama_prebuilt/**/llama-server", recursive=True):
    server_bin = path
    break

if not server_bin:
    raise FileNotFoundError("解压失败，未找到 llama-server 执行文件！")

# Start llama.cpp server in the background
print(f"Starting {server_bin}...")
server_cmd = f"{server_bin} -m Sakura-Galtransl-14B-v3-Q5_K_S.gguf -c 4096 --host 127.0.0.1 --port 8000 -ngl 999"
subprocess.Popen(server_cmd, shell=True, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

time.sleep(10)  # Wait for the model to load in VRAM
print("Server is running locally at http://127.0.0.1:8000")

## 3. Batch Translate EPUBs
Put your `.epub` files inside the `books/` folder in Colab before running this cell. The translated files will be saved in `translated/`.

In [ ]:
!mkdir -p books
!mkdir -p translated

!echo "Start batch translation..."
!epub-translator ./books/ ./translated/ --provider custom --model sakura-14b --api-url http://127.0.0.1:8000/v1/chat/completions

!echo "Translation finished. Check the 'translated' folder!"